In [ ]:
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Maintenance Scheduling

This tutorial demonstrates maintenance scheduling for generators. Setting `maintainable=True` introduces binary variables that schedule contiguous maintenance blocks, reducing or eliminating the component's available capacity during those periods.

To enable maintenance scheduling on a component (`Link` or `Generator`), set its attribute `maintainable=True` and specify `maintenance_duration`.

In [ ]:
import matplotlib.pyplot as plt

import pypsa

## Basic Maintenance Scheduling

A cheap generator must undergo 3 snapshots of maintenance. The optimizer places the maintenance block where the backup generator can cover the load at minimum cost.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=3,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)

load = [80] * 10
load[0] = 10
load[1] = 10
load[2] = 10
n.add("Load", "load", bus="bus", p_set=load)

In [ ]:
n.optimize()

The maintenance is scheduled during the low-demand period (snapshots 0-2), minimizing the cost of dispatching the expensive backup generator:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Partial Maintenance

With `maintenance_pu=0.5`, only 50% of capacity is unavailable during maintenance. The generator can still dispatch at reduced capacity.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=2,
    maintenance_pu=0.5,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=80)

In [ ]:
n.optimize()

During maintenance, the cheap generator is limited to 50 MW (50% of 100 MW). The backup covers the remaining 30 MW of the 80 MW load:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Partial Maintenance)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Multiple Events

With `maintenance_events=2`, the generator must undergo two separate maintenance blocks of `maintenance_duration=2` snapshots each.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

n.add(
    "Generator",
    "cheap",
    bus="bus",
    p_nom=100,
    marginal_cost=10,
    maintainable=True,
    maintenance_duration=2,
    maintenance_events=2,
)

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=50)

In [ ]:
n.optimize()

Both maintenance blocks are contiguous, and the total maintenance duration is 4 snapshots (2 events x 2 snapshots):

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Multiple Events)")

n.generators_t.maintenance["cheap"].plot.bar(ax=axes[1], color="C3")
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()

## Staggered Maintenance for Multiple Generators

When multiple generators require maintenance, the optimizer will stagger their maintenance windows to maintain system adequacy. Here, two generators each need 2-snapshot maintenance, but the load requires at least one to be online at all times.

In [ ]:
n = pypsa.Network(snapshots=range(10))

n.add("Bus", "bus")

for i in range(2):
    n.add(
        "Generator",
        f"gen{i}",
        bus="bus",
        p_nom=100,
        marginal_cost=10,
        maintainable=True,
        maintenance_duration=2,
    )

n.add("Generator", "backup", bus="bus", p_nom=100, marginal_cost=100)
n.add("Load", "load", bus="bus", p_set=150)

In [ ]:
n.optimize()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

n.generators_t.p.plot.bar(stacked=True, ax=axes[0], legend=True)
axes[0].set_ylabel("Dispatch [MW]")
axes[0].set_title("Generator Dispatch (Staggered Maintenance)")

n.generators_t.maintenance[["gen0", "gen1"]].plot.bar(ax=axes[1])
axes[1].set_ylabel("Maintenance")
axes[1].set_title("Maintenance Status")
axes[1].set_xlabel("Snapshot")

plt.tight_layout()